# Analyze customer reviews to identify key topics and sentiments without noise from irrelevant words.

In [ ]:
!pip install -q datasets nltk scikit-learn pandas

In [ ]:
import re
from collections import Counter

import nltk
import pandas as pd
from datasets import load_dataset
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("stopwords", quiet=True)
nltk.download("vader_lexicon", quiet=True)

N_REVIEWS = 2000   # sample size, keeps Colab fast; raise it if you like
SEED = 42

In [ ]:
# ------------------------------------------------------------
# 1. Load the dataset (balanced random sample)
# ------------------------------------------------------------
ds = load_dataset("fancyzhx/yelp_polarity", split="train")
df = ds.to_pandas()
df = (df.groupby("label", group_keys=False)
        .apply(lambda g: g.sample(N_REVIEWS // 2, random_state=SEED))
        .reset_index(drop=True))

print("Sample shape:", df.shape)
print(df["label"].value_counts().rename({0: "negative", 1: "positive"}))
print("\nExample review:\n", df["text"].iloc[0][:300], "...")

README.md:   0%|          | 0.00/8.93k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  256MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.7MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

Sample shape: (2000, 2)
label
negative    1000
positive    1000
Name: count, dtype: int64

Example review:
 I do enjoy Taco Bell from time to time but I hate to say it but the last few times its been forgettable , had one incident of a decent size piece of plastic my 4 yr old son was choking on from his order cooked Into his Mexican pizza ...


/tmp/ipykernel_4370/2569517470.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(N_REVIEWS // 2, random_state=SEED))


In [ ]:
# ------------------------------------------------------------
# 2. Cleaning helpers
# ------------------------------------------------------------
def tokenize(text):
    text = text.replace("\\n", " ").lower()   # Yelp text contains literal "\n"
    return re.findall(r"[a-z']+", text)

# NLTK's stopword list contains negations ("not", "no", "didn't", ...).
# Removing them would flip "not tasty" -> "tasty", so we keep them.
stop_words = set(stopwords.words("english"))
keep = {"not", "no", "nor", "never"}
stop_words = {w for w in stop_words if w not in keep and "n't" not in w}

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

df["tokens_raw"] = df["text"].apply(tokenize)
df["tokens_clean"] = df["tokens_raw"].apply(remove_stopwords)
df["text_raw"] = df["tokens_raw"].apply(" ".join)
df["text_clean"] = df["tokens_clean"].apply(" ".join)

In [ ]:
# ------------------------------------------------------------
# STEP 1 vs STEP 2: word frequencies
# ------------------------------------------------------------
raw_counts = Counter(t for toks in df["tokens_raw"] for t in toks)
clean_counts = Counter(t for toks in df["tokens_clean"] for t in toks)

print("\n" + "=" * 70)
print("STEP 1: Top 15 words WITHOUT stopword removal")
print("=" * 70)
print(raw_counts.most_common(15))

print("\n" + "=" * 70)
print("STEP 2: Top 15 words WITH stopword removal")
print("=" * 70)
print(clean_counts.most_common(15))

n_raw = sum(raw_counts.values())
n_clean = sum(clean_counts.values())
print(f"\nTokens before: {n_raw:,} | after: {n_clean:,} "
      f"| removed: {n_raw - n_clean:,} ({(n_raw - n_clean) / n_raw:.0%})")

# Most distinctive words per sentiment class (after cleaning)
pos_counts = Counter(t for toks in df.loc[df.label == 1, "tokens_clean"] for t in toks)
neg_counts = Counter(t for toks in df.loc[df.label == 0, "tokens_clean"] for t in toks)

print("\nTop 12 words in POSITIVE reviews (cleaned):", pos_counts.most_common(12))
print("Top 12 words in NEGATIVE reviews (cleaned):", neg_counts.most_common(12))


STEP 1: Top 15 words WITHOUT stopword removal
[('the', 14293), ('and', 8964), ('i', 7754), ('a', 7098), ('to', 7093), ('was', 4930), ('of', 4124), ('it', 3649), ('in', 3231), ('for', 3208), ('is', 3066), ('that', 2692), ('my', 2417), ('we', 2340), ('this', 2247)]

STEP 2: Top 15 words WITH stopword removal
[('not', 1848), ('food', 1211), ('place', 1167), ('good', 1061), ('like', 985), ('get', 891), ('one', 878), ('service', 851), ('would', 822), ('great', 819), ('time', 801), ('back', 780), ('no', 746), ('go', 734), ('us', 579)]

Tokens before: 275,799 | after: 141,181 | removed: 134,618 (49%)

Top 12 words in POSITIVE reviews (cleaned): [('good', 609), ('great', 608), ('place', 577), ('food', 531), ('not', 524), ('like', 454), ('one', 366), ('time', 359), ('get', 355), ('go', 313), ('service', 300), ('back', 294)]
Top 12 words in NEGATIVE reviews (cleaned): [('not', 1324), ('food', 680), ('place', 590), ('no', 563), ('would', 556), ('service', 551), ('get', 536), ('like', 531), ('one

In [ ]:
# ------------------------------------------------------------
# IMPACT 1: Sentiment analysis (VADER) vs the true labels
# ------------------------------------------------------------
sia = SentimentIntensityAnalyzer()

def vader_pred(text):
    return int(sia.polarity_scores(text)["compound"] >= 0)

df["pred_raw"] = df["text_raw"].apply(vader_pred)
df["pred_clean"] = df["text_clean"].apply(vader_pred)

acc_raw = (df["pred_raw"] == df["label"]).mean()
acc_clean = (df["pred_clean"] == df["label"]).mean()

print("\n" + "=" * 70)
print("IMPACT 1: VADER sentiment accuracy against true Yelp labels")
print("=" * 70)
print(f"Without stopword removal: {acc_raw:.3f}")
print(f"With stopword removal:    {acc_clean:.3f}")
print("(VADER already handles function words like 'but' and 'very' itself,\n"
      " so its accuracy can move either way; report whatever you observe.)")


IMPACT 1: VADER sentiment accuracy against true Yelp labels
Without stopword removal: 0.706
With stopword removal:    0.688
(VADER already handles function words like 'but' and 'very' itself,
 so its accuracy can move either way; report whatever you observe.)


In [ ]:
# ------------------------------------------------------------
# IMPACT 2: Summarization-style keywords
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("IMPACT 2: Top-5 keywords per sample review (cleaned)")
print("=" * 70)
for i in range(3):
    kws = [w for w, _ in Counter(df["tokens_clean"].iloc[i]).most_common(5)]
    label = "positive" if df["label"].iloc[i] else "negative"
    print(f"[{label}] {kws}")


IMPACT 2: Top-5 keywords per sample review (cleaned)
[negative] ['time', 'enjoy', 'taco', 'bell', 'hate']
[negative] ['waited', 'salty', 'eat', 'meals', 'guest']
[negative] ['u', 'e', 'four', 'le', 'vrai']


In [ ]:
# ------------------------------------------------------------
# IMPACT 3: Topic modeling (LDA), same settings except stopwords
# ------------------------------------------------------------
def lda_topics(texts, n_topics=5, n_words=8):
    vec = CountVectorizer(min_df=5, max_df=0.5, max_features=5000)
    X = vec.fit_transform(texts)
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=SEED,
                                    learning_method="batch", max_iter=15)
    lda.fit(X)
    vocab = vec.get_feature_names_out()
    return [[vocab[j] for j in comp.argsort()[-n_words:][::-1]]
            for comp in lda.components_]

print("\n" + "=" * 70)
print("IMPACT 3a: LDA topics WITHOUT stopword removal")
print("=" * 70)
for i, words in enumerate(lda_topics(df["text_raw"]), 1):
    print(f"Topic {i}: {', '.join(words)}")

print("\n" + "=" * 70)
print("IMPACT 3b: LDA topics WITH stopword removal")
print("=" * 70)
for i, words in enumerate(lda_topics(df["text_clean"]), 1):
    print(f"Topic {i}: {', '.join(words)}")


IMPACT 3a: LDA topics WITHOUT stopword removal
Topic 1: you, good, have, they, had, so, food, as
Topic 2: we, our, were, had, not, at, us, food
Topic 3: she, me, her, not, have, he, so, as
Topic 4: great, are, place, you, food, service, good, love
Topic 5: they, you, have, at, not, me, are, there

IMPACT 3b: LDA topics WITH stopword removal
Topic 1: not, no, us, would, back, service, room, get
Topic 2: get, great, like, not, time, place, don, one
Topic 3: food, good, not, place, great, like, one, ordered
Topic 4: place, good, get, great, go, like, vegas, always
Topic 5: not, food, service, place, like, time, one, good


In [ ]:
# ============================================================
# RESULT SUMMARY
# Dataset: Yelp Polarity (2,000-review balanced sample)
# ============================================================

# 1. Sentiment Analysis (VADER vs true Yelp labels)
#    - Without stopword removal: 0.706 accuracy
#    - With stopword removal:    0.688 accuracy
#    Observation: Removing stopwords slightly REDUCED accuracy (~2 points).
#    VADER is a lexicon/rule-based model that relies on function words
#    such as "very", "so", "too" and "but" as intensifiers and contrast
#    markers. Removing them discards signal it uses. Negations (not, no,
#    n't) were deliberately kept to avoid flipping meaning.
#    Conclusion: Stopword removal does not automatically improve
#    sentiment analysis; the effect depends on the model used.

# 2. Topic Modeling (LDA, 5 topics)
#    - Without stopword removal: topics were dominated by pronouns and
#      function words (you/they/have, we/our/us/were, she/he/her/me).
#      LDA grouped reviews by WHO was speaking rather than by subject,
#      so the topics were not meaningful.
#    - With stopword removal: topics contained content words such as
#      food, service, room, ordered, place and vegas, giving themes like
#      food and ordering, service and waiting time, and hotel stays.
#    Conclusion: Stopword removal clearly improved topic interpretability.

# 3. Remaining Limitations
#    - "not"/"no" appear in almost every cleaned topic; they help sentiment
#      but add noise to topic modeling, so they should be dropped for LDA.
#    - Generic words (place, good, great, like, one, get) still blur topics;
#      a custom domain stopword list would help.
#    - "don" is an artifact from splitting "don't" in the vectorizer.
#    - Yelp includes non-restaurant businesses (e.g. hotels), which adds
#      topics unrelated to food.

# OVERALL CONCLUSION
# Stopword removal is most beneficial for topic modeling and keyword
# extraction, where it removes noise and exposes meaningful themes. For
# lexicon-based sentiment analysis (VADER) it gave no improvement and
# slightly hurt accuracy, so it should be applied selectively.
# ============================================================